In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os

In [5]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, 4),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [11]:
full_train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root = './data',
    train = False,
    download=True,
    transform=test_transform
)


In [23]:
train_size = int(len(full_train_dataset)*0.9)
validation_size = len(full_train_dataset) - train_size
print(train_size)
print(validation_size)
print(len(test_dataset))
train_dataset, validation_dataset = random_split(
    full_train_dataset, 
    [train_size, validation_size]
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=128,
    shuffle = True,
    num_workers=4,
    pin_memory=True
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


45000
5000
10000


In [29]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        self.apply(self.init_w)

    def init_w(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    


In [34]:
model = CNN(num_classes=10)
print(sum(p.numel() for p in model.parameters()))
print(sum(p.numel() for p in model.parameters() if p.requires_grad))

1148938
1148938


In [41]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = 'min',
    factor = 0.2,
    patience = 3,
    verbose=True
)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    rls = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        rls += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = rls / total
    epoch_acc = 100 * correct / total

    return epoch_acc, epoch_loss

def validate_epoch(model, loader, criterion, device):
    model.eval()
    rls = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            rls += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = rls / total
    epoch_acc = 100 * correct / total

    return epoch_acc, epoch_loss




c:\Users\teckk\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)

n_epoch = 50
best_val_acc = 0.0
train_losses, val_losses, train_accs, val_accs = [],[],[],[]

for epoch in range(n_epoch):
    train_acc, train_loss = train_epoch(model, train_dataloader, criterion, optimizer, device)
    val_acc, val_loss = validate_epoch(model, validation_dataloader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    scheduler.step(val_loss)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_cnn.pth')
        save_msg = 'фаин'
    else:
        save_msg = ''

    print(f"Эпоха {epoch+1:2d}/{n_epoch} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% {save_msg}")

cuda
Эпоха  1/50 | Train Loss: 1.2600 | Train Acc: 54.50% | Val Loss: 1.2591 | Val Acc: 55.16% фаин
Эпоха  2/50 | Train Loss: 1.1609 | Train Acc: 58.32% | Val Loss: 1.1223 | Val Acc: 60.48% фаин
Эпоха  3/50 | Train Loss: 1.0897 | Train Acc: 60.87% | Val Loss: 1.1648 | Val Acc: 59.24% 
Эпоха  4/50 | Train Loss: 1.0325 | Train Acc: 63.13% | Val Loss: 0.9911 | Val Acc: 64.58% фаин
Эпоха  5/50 | Train Loss: 0.9863 | Train Acc: 65.03% | Val Loss: 0.9307 | Val Acc: 67.54% фаин
Эпоха  6/50 | Train Loss: 0.9481 | Train Acc: 66.33% | Val Loss: 0.9106 | Val Acc: 66.64% 
Эпоха  7/50 | Train Loss: 0.9222 | Train Acc: 67.39% | Val Loss: 0.8140 | Val Acc: 70.76% фаин
Эпоха  8/50 | Train Loss: 0.8938 | Train Acc: 68.52% | Val Loss: 0.7666 | Val Acc: 72.80% фаин
Эпоха  9/50 | Train Loss: 0.8652 | Train Acc: 69.31% | Val Loss: 0.8572 | Val Acc: 69.62% 
Эпоха 10/50 | Train Loss: 0.8386 | Train Acc: 70.42% | Val Loss: 0.7430 | Val Acc: 74.30% фаин
Эпоха 11/50 | Train Loss: 0.8250 | Train Acc: 71.09% | Va